In [1]:
import torch
import torch.nn as nn

torch.manual_seed(42)

vocab_size = 20
seq_len = 5
embed_dim = 8
num_heads = 2
num_classes = 2

x = torch.tensor([
    [1,2,3,4,5],
    [6,7,8,9,10],
    [11,12,13,14,15]
])

embeddings = nn.Embedding(vocab_size,embed_dim)
x_embed = embeddings(x)


print("token embedding shape -> ",x_embed.shape)

token embedding shape ->  torch.Size([3, 5, 8])


In [2]:
x_embed[0]

tensor([[-0.7521,  1.6487, -0.3925, -1.4036, -0.7279, -0.5594, -0.7688,  0.7624],
        [ 1.6423, -0.1596, -0.4974,  0.4396, -0.7581,  1.0783,  0.8008,  1.6806],
        [ 1.2791,  1.2964,  0.6105,  1.3347, -0.2316,  0.0418, -0.2516,  0.8599],
        [-1.3847, -0.8712, -0.2234,  1.7174,  0.3189, -0.4245,  0.3057, -0.7746],
        [-1.5576,  0.9956, -0.8798, -0.6011, -1.2742,  2.1228, -1.2347, -0.4879]],
       grad_fn=<SelectBackward0>)

In [3]:
positional_embeddings = nn.Embedding(seq_len,embed_dim)
positions = torch.arange(seq_len)
pos_embed = positional_embeddings(positions)

x_embed = x_embed + pos_embed

x_embed[0]

tensor([[-2.6527,  1.8773, -0.3676, -1.7496, -0.4410, -1.2903, -0.5940, -0.3315],
        [ 0.0402,  1.1933,  0.7914,  0.4919, -2.3050,  1.8350,  1.5763,  3.7072],
        [ 1.3149,  1.4170, -0.1952,  1.1272, -1.1636, -1.5492, -1.3876,  0.3373],
        [-1.5440, -1.2962,  0.7209,  1.5324,  1.3797, -0.2162, -0.2721, -0.4491],
        [-1.2958,  0.2357, -2.9259, -2.1306, -0.8693,  2.7547, -0.9221, -0.5214]],
       grad_fn=<SelectBackward0>)

In [4]:
pos_embed[0]

tensor([-1.9006,  0.2286,  0.0249, -0.3460,  0.2868, -0.7308,  0.1748, -1.0939],
       grad_fn=<SelectBackward0>)

In [5]:
# Transformers

encoder_layer = nn.TransformerEncoderLayer(
    d_model = embed_dim,
    nhead = num_heads,
    dim_feedforward = 32,
    dropout = 0.1,
    batch_first = True
)

transformer_encoder = nn.TransformerEncoder(
    encoder_layer = encoder_layer,
    num_layers = 2
)

x_out = transformer_encoder(x_embed)

print("Transformer output shape -> ",x_out.shape)

Transformer output shape ->  torch.Size([3, 5, 8])


In [6]:
# Mean Pooling

x_pooled = x_out.mean(dim=1)

print("Pooled Output : ",x_pooled.shape)

Pooled Output :  torch.Size([3, 8])


In [7]:
# Classification head

classification_head = nn.Linear(embed_dim,num_classes)

logits = classification_head(x_pooled)

print("Logits shape -> ",logits.shape)

prediction = logits.argmax(dim=1)

print("Prediction : ",prediction)


Logits shape ->  torch.Size([3, 2])
Prediction :  tensor([0, 1, 1])


In [8]:
# full trainable model

class SimpleTransformer(nn.Module):
  def __init__(self):
    super().__init__()

    self.token_embedding = nn.Embedding(20,8)
    self.pos_embedding = nn.Embedding(5,8)

    layer = nn.TransformerEncoderLayer(
        d_model = 8,
        nhead = 2,
        dim_feedforward = 32,
        dropout = 0.1,
        batch_first = True
    )

    self.encoder = nn.TransformerEncoder(
        encoder_layer = layer,
        num_layers = 2
    )

    self.classification_head = nn.Linear(8,2)

  def forward(self,x):
    x_embed = self.token_embedding(x)

    # Fix: Create positions on the same device as input x
    positions = torch.arange(x.shape[1], device=x.device)
    pos_embed = self.pos_embedding(positions)

    x_embed = x_embed + pos_embed

    x_out = self.encoder(x_embed)

    x_pooled = x_out.mean(dim=1)

    logits = self.classification_head(x_pooled)
    return logits

In [10]:
device = torch.device("mps" if torch.mps.is_available() else "cpu")
model = SimpleTransformer()
model = model.to(device)
print(model)
print(model.cuda)


SimpleTransformer(
  (token_embedding): Embedding(20, 8)
  (pos_embedding): Embedding(5, 8)
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=8, out_features=8, bias=True)
        )
        (linear1): Linear(in_features=8, out_features=32, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=32, out_features=8, bias=True)
        (norm1): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((8,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (classification_head): Linear(in_features=8, out_features=2, bias=True)
)
<bound method Module.cuda of SimpleTransformer(
  (token_embedding): Embedding(20, 8)
  (pos_embedding): Embedding(5, 8